In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv',header=[0,1])
df

1. 컬럼명 공백제거
2. 중복 제거
3. defects 2번, 3번을 불량(1)로 대체
4. 컬럼 모두 소문자
5. id, shot 컬럼 drop
기타 - Factory Nan 값 drop or median 어떻게?

======================================= product_type 나눈 시점

6. product_type 컬럼 삭제
7. (product_type별로) 0값만 있는 defects 컬럼 drop

In [ ]:
### 1. 컬럼명 공백 제거
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

In [ ]:
### 2. 중복 제거
# 제외할 컬럼 리스트 정의
exclude_cols = [('Process', 'id')]

# 전체 컬럼에서 제외할 컬럼만 뺀 리스트 생성
target_cols = df.columns.difference(exclude_cols).tolist()

# 중복 제거 실행 (첫 번째 행은 남기고 나머지는 삭제)
df_cleaned = df.drop_duplicates(subset=target_cols, keep='first').reset_index(drop=True)

# 결과 확인
print(f"제거 전 행 개수: {len(df)}")
print(f"제거 후 행 개수: {len(df_cleaned)}")
print(f"삭제된 중복 행 개수: {len(df) - len(df_cleaned)}")

In [ ]:
### 3. 2,3을 불량 1로 대체
defect_cols = df_cleaned['Defects'].columns

# 'Defects' 대분류 아래의 모든 컬럼에 대해 2 이상의 값을 1로 변경
for col in defect_cols:
    # Defects 하위의 col 값 중 2 이상인 경우를 1로 치환
    df_cleaned.loc[df_cleaned[('Defects', col)] >= 2, ('Defects', col)] = 1

In [ ]:
### 4. factory NaN 값 제거?
# 컬럼별 결측치 개수 합계
print(df_cleaned.isnull().sum())

In [ ]:
df_cleaned

In [ ]:
### 5. product_type 별 데이터 분리
product_cols = [c for c in df_cleaned.columns if str(c[1]).strip().lower() == "product_type"]
if not product_cols:
    raise ValueError("Product_Type 컬럼을 찾지 못했습니다. df.columns를 확인하세요.")

product_col = product_cols[0]

print("Product_Type 컬럼:", product_col)
print(df_cleaned[product_col].value_counts(dropna=False))

In [ ]:
### 6. Product_Type별 불량 컬럼 불량률 확인
product_col = ('Process', 'Product_Type')

defects_df = df_cleaned['Defects']          # <- 여기서 Defects 아래만 DataFrame으로 분리
defect_rate = defects_df.groupby(df_cleaned[product_col]).mean()

print(defect_rate.head())

In [ ]:
# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 전체 평균 기준 Top10 불량
top_defects = defect_rate.mean().sort_values(ascending=False).head(10).index

# 그래프
plt.figure(figsize=(18,6))

defect_rate[top_defects].T.plot(kind='bar')

plt.title("Product Type별 불량률 Top 10")
plt.ylabel("Defect 주기")
plt.xlabel("Defect 유형")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 컬럼별 결측치 개수 합계
print(df_cleaned.isnull().sum())

In [ ]:
# 7. Product_Type이 1인 행만 필터링하여 저장
df_type_1 = df_cleaned[df_cleaned[('Process', 'Product_Type')] == 1].reset_index(drop=True)
df_type_1.to_csv('product_type_1.csv', index=False, encoding='utf-8-sig')

# 8. Product_Type이 2인 행만 필터링하여 저장
df_type_2 = df_cleaned[df_cleaned[('Process', 'Product_Type')] == 2].reset_index(drop=True)
df_type_2.to_csv('product_type_2.csv', index=False, encoding='utf-8-sig')

In [ ]:
df1 = pd.read_csv('product_type_1.csv',header=[0,1])
df2 = pd.read_csv('product_type_2.csv',header=[0,1])

In [ ]:
# 컬럼별 결측치 개수 합계
print(df1.isnull().sum())

In [ ]:
# 컬럼별 결측치 개수 합계
print(df2.isnull().sum())